# ブラケット（Bra-Ket）記法 完全チュートリアル


このノートブックは、ディラック（Dirac）によって導入されたブラケット記法を理論と実装の両面から学習するための包括的なガイドです。

1. 抽象ベクトル空間としてのブラケット記法の理解
2. 内積・外積・射影演算子の計算
3. エルミート演算子と固有値問題
4. Python/Numpyによる数値実装
5. スピン1/2系への応用

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from braket_notation import (
    Ket, Bra, Operator,
    outer_product, projection_operator,
    QuantumStates, PauliMatrices, BlochSphere
)

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (10, 6)

## 第1章：基本概念


量子状態は**ケット** $|\psi\rangle$ で表されます。これはヒルベルト空間の列ベクトルです：

$$
|\psi\rangle = \begin{pmatrix} \psi_1 \\ \psi_2 \end{pmatrix}
$$

その**エルミート共役**（複素共役転置）を**ブラ** $\langle\psi|$ と呼びます：

$$
\langle\psi| = (\psi_1^*, \psi_2^*)
$$

In [ ]:
# ケットの作成
psi = Ket([1, 1j])
print("ケット |ψ⟩:")
print(psi)
print(f"\n状態ベクトル: {psi.state}")

bra_psi = psi.bra()
print("\nブラ ⟨ψ|:")
print(bra_psi)
print(f"\n複素共役: {bra_psi.state.conj()}")

### 1.2 正規化

量子状態は通常、**正規化**されています：

$$
\langle\psi|\psi\rangle = 1
$$

これは確率の総和が1であることを保証します。

In [ ]:
# 正規化されていない状態
unnormalized = Ket([3, 4])
print(f"正規化前: {unnormalized.state}")
print(f"ノルム: {np.sqrt(np.vdot(unnormalized.state, unnormalized.state).real):.4f}")

normalized = unnormalized.normalize()
print(f"\n正規化後: {normalized.state}")
print(f"ノルム: {np.sqrt(np.vdot(normalized.state, normalized.state).real):.4f}")
print(f"正規化確認: {normalized.is_normalized()}")

## 第2章：内積と外積


二つの状態の**内積**は：

$$
\langle\phi|\psi\rangle = \phi_1^* \psi_1 + \phi_2^* \psi_2
$$

物理的意味：
- $|\langle\phi|\psi\rangle|^2$ は、状態 $|\psi\rangle$ を測定して $|\phi\rangle$ を得る確率
- 直交状態では $\langle\phi|\psi\rangle = 0$

In [ ]:
# スピン状態の定義
spin_up = QuantumStates.spin_up()
spin_down = QuantumStates.spin_down()

print("スピン上 |↑⟩:", spin_up.state)
print("スピン下 |↓⟩:", spin_down.state)

inner_same = spin_up.bra() * spin_up
inner_ortho = spin_up.bra() * spin_down

print(f"\n⟨↑|↑⟩ = {inner_same}")
print(f"⟨↑|↓⟩ = {inner_ortho}")

plus_state = QuantumStates.plus_state()
print(f"\nプラス状態 |+⟩ = (|↑⟩ + |↓⟩)/√2 = {plus_state.state}")

prob_up = abs(spin_up.bra() * plus_state)**2
prob_down = abs(spin_down.bra() * plus_state)**2

print(f"\n|+⟩を測定して|↑⟩を得る確率: {prob_up:.4f}")
print(f"|+⟩を測定して|↓⟩を得る確率: {prob_down:.4f}")
print(f"確率の和: {prob_up + prob_down:.4f}")

## 第3章：演算子と観測量


スピン1/2系の基本的な演算子は**パウリ行列**です：

$$
\sigma_x = \begin{pmatrix}0 & 1 \\ 1 & 0\end{pmatrix}, \quad
\sigma_y = \begin{pmatrix}0 & -i \\ i & 0\end{pmatrix}, \quad
\sigma_z = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}
$$

In [ ]:
# パウリ行列の定義
sigma_x = PauliMatrices.sigma_x()
sigma_y = PauliMatrices.sigma_y()
sigma_z = PauliMatrices.sigma_z()

print("σₓ:")
print(sigma_x.matrix)
print(f"\nエルミート: {sigma_x.is_hermitian()}")

eigenvals, eigenvecs = sigma_z.eigenvalues_eigenvectors()
print("\nσᵤの固有値と固有ベクトル:")
for i, (val, vec) in enumerate(zip(eigenvals, eigenvecs)):
    print(f"λ_{i} = {val:+.4f}, |ψ_{i}⟩ = {vec.state}")

## 第4章：ブロッホ球表現

スピン1/2の任意の純粋状態は、ブロッホ球上の点として表現できます。

ブロッホベクトル：

$$
\vec{r} = (\langle\sigma_x\rangle, \langle\sigma_y\rangle, \langle\sigma_z\rangle)
$$

In [ ]:
# ブロッホベクトルの計算
states = [
    ("Spin up |↑⟩", QuantumStates.spin_up()),
    ("Plus |+⟩", QuantumStates.plus_state()),
    ("Right |R⟩", QuantumStates.right_circular()),
]

print(f"{'状態':<20} {'ブロッホベクトル (x, y, z)':<35}")
print("-" * 60)

for name, state in states:
    bloch_vec = BlochSphere.state_to_bloch_vector(state)
    print(f"{name:<20} ({bloch_vec[0]:>6.3f}, {bloch_vec[1]:>6.3f}, {bloch_vec[2]:>6.3f})")

state_list = [state for _, state in states]
labels = [name for name, _ in states]
fig = BlochSphere.plot_bloch_sphere(state_list, labels, "ブロッホ球")
plt.show()

## まとめ

このチュートリアルでは、ブラケット記法の基礎から応用まで学習しました：

1. ケットとブラの定義と性質
2. 内積と外積の計算
3. 演算子と期待値
4. パウリ行列と固有値問題
5. ブロッホ球表現

これらの概念は、量子力学、量子情報、量子計算の基礎となります。


1. 正規直交基底の確認
2. 射影演算子の冪等性の証明
3. エルミート演算子の期待値が実数であることの確認